## First milestone

in this code file, we are going to choose our tasks (e.g., motor imagery + P300 + SSVEP) and state precisely what 'generalizes to a new subject' means and how you'll measure it.

In [1]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.pipeline import make_pipeline

import moabb
from moabb.datasets import BNCI2014_001
from moabb.evaluations import WithinSessionEvaluation
from moabb.paradigms import LeftRightImagery

moabb.set_log_level("info")
warnings.filterwarnings("ignore")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = BNCI2014_001()
dataset.subject_list = [1,2,3,4,5]

In [3]:
sessions = dataset.get_data(subjects=[1,2,3,4])

100%|█████████████████████████████████████| 37.2M/37.2M [00:00<00:00, 75.8GB/s]
SHA256 hash of downloaded file: 15850d81b95fc88cc8b9589eb9b713d49fa071e28adaf32d675b3eaa30591d6e
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
100%|█████████████████████████████████████| 41.7M/41.7M [00:00<00:00, 22.6GB/s]
SHA256 hash of downloaded file: 81916dff2c12997974ba50ffc311da006ea66e525010d010765f0047e771c86a
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


After initializating out dataset and subjects, now we will jump to our task. 

In this task, we will training on subjects 1, 2, 3, 4 and train on subject 5

In [8]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

paradigm = LeftRightImagery()

# import subjects 1,2,3,4 for training
X_train, y_train, meta_train = paradigm.get_data(
    dataset=dataset, 
    subjects=[1,2,3,4]
)

# import the 5th subject for testing
X_test, y_test, meta_test = paradigm.get_data(
    dataset=dataset, 
    subjects=[5]
)

print ("X_train shape: ", X_train.shape)
print ("y_train shape: ", y_train.shape)
print ("X_test shape: ", X_test.shape)
print ("y_test shape: ", y_test.shape)

pipeline = make_pipeline(
    CSP(n_components=8, reg=None, log=True, norm_trace=False), 
    LDA()
)

# now we will train the model on the first 4 subjects
pipeline.fit(X_train, y_train)

# test on subject-5
y_pred = pipeline.predict(X_test)

# evaluate
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print ("Accuracy: ", acc)
print ("Balanced Accuracy: ", bal_acc)

print ("Classification Report:")
print (classification_report(y_test, y_pred))

print ("Confusion Matrix:")
print (confusion_matrix(y_test, y_pred))

X_train shape:  (1152, 22, 1001)
y_train shape:  (1152,)
X_test shape:  (288, 22, 1001)
y_test shape:  (288,)
Accuracy:  0.5694444444444444
Balanced Accuracy:  0.5694444444444444
Classification Report:
              precision    recall  f1-score   support

   left_hand       0.54      0.86      0.67       144
  right_hand       0.67      0.28      0.39       144

    accuracy                           0.57       288
   macro avg       0.61      0.57      0.53       288
weighted avg       0.61      0.57      0.53       288

Confusion Matrix:
[[124  20]
 [104  40]]
